## Notebook: Project exploration code.

#### Packages and assumptions

In [20]:
# NumPy released version 2.0 and removed some old aliases: np.float_, np.complex_, np.int_. 
# The Alexandria library was written for the old NumPy and still tries to use those names. When it does, Python crashes.
# This patch adds the missing names back before Alexandria loads.

import numpy as _np_tmp
if not hasattr(_np_tmp, 'float_'):   _np_tmp.float_   = _np_tmp.float64
if not hasattr(_np_tmp, 'complex_'): _np_tmp.complex_ = _np_tmp.complex128
if not hasattr(_np_tmp, 'int_'):     _np_tmp.int_     = _np_tmp.int64
if not hasattr(_np_tmp, 'bool_'):    _np_tmp.bool_    = _np_tmp.bool_.__class__  # already exists as bool
del _np_tmp

# Packages to import
import warnings; warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.api import VAR
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.regression.linear_model import OLS
from alexandria import MinnesotaBayesianVar
import alexandria.vector_autoregression.var_utilities as vu

In [ ]:
# EXPLORATION SPEC
# Exports-first ordering used in Phase 3/4/5
ENDOG_OLD = ['hk_exports_china_yoy', 'gdp_growth', 'cpi_inflation',
             'unemployment', 'hibor_3m', 'hk_property_price_qoq']
AR_COEF_OLD = [0.627, 0.545, 0.735, 0.991, 0.442, 0.418]
# order:        exp    gdp   cpi   unemp  hibor  prop

EXOG = ['us_ffr', 'china_gdp']
LAGS = 4
PI1, PI2, PI3 = 0.085, 1.0, 1.0
PI4  = 100.0
DATA = 'data/hk_macro_varx_ready.csv'

FINAL SPEC loaded.
  ENDOG:   ['hibor_3m', 'hk_exports_china_yoy', 'hk_property_price_qoq', 'gdp_growth', 'cpi_inflation', 'unemployment']
  EXOG:    ['us_ffr', 'china_gdp']
  LAGS:    4
  pi1=0.085  pi2=1.0  pi3=1.0  pi4=100.0
  AR_COEF: [0.442, 0.627, 0.418, 0.545, 0.735, 0.991]
           hibor  exp    prop  gdp   cpi   unemp


## VARX(1) Frequentist Baseline

Spec: VARX(1), endog=ENDOG_OLD (exports-first), exog=EXOG
Data: data/hk_macro_varx_ready.csv, 113 rows, property_qoq (I(0))
CIs: Bootstrap MC, 1000 replications, 90%

Key findings:
- us_ffr to hibor_3m: +0.530 (p<0.001), LERS confirmed
- china_gdp to exports: +0.806 (p=0.016), trade channel confirmed
- hibor to property (h=1): (-1.67, -0.29), significant
- exports to gdp (h=1): (+0.15, +0.91), significant, crosses 0 at h=2, recovered by BVAR
- 2/6 Ljung-Box failures: gdp_growth, cpi_inflation, structural breaks not misspecification


In [5]:
# Phase 3: VARX(1) — frequentist baseline

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
exog   = df[EXOG]
endog3 = df[ENDOG_OLD]
model3 = VAR(endog3, exog=exog)
v1     = model3.fit(maxlags=1)

print(f'VARX(1) | obs={v1.nobs} | AIC={v1.aic:.4f} | BIC={v1.bic:.4f}')
print()

# Ljung-Box
print('Ljung-Box (lag=8):')
for col in ENDOG_OLD:
    p = acorr_ljungbox(v1.resid[col], lags=[8], return_df=True)['lb_pvalue'].values[0]
    print(f'  {col:<32} p={p:.4f}  {"FAIL" if p<0.05 else "pass"}')

print()
# Exog transmission check
print('Exogenous coefficients (LERS + trade channel):')
key_pairs = [
    ('us_ffr',    'hibor_3m',            'us_ffr → hibor (LERS)'),
    ('china_gdp', 'hk_exports_china_yoy','china_gdp → exports (trade)'),
]
for exog_var, eq, label in key_pairs:
    coef = v1.params.loc[exog_var, eq]
    se   = v1.stderr.loc[exog_var, eq]
    tstat = coef / se
    pval  = 2*(1 - stats.t.cdf(abs(tstat), df=v1.nobs - v1.k_ar*len(ENDOG_OLD)))
    sig   = '***' if pval<0.01 else ('**' if pval<0.05 else ('*' if pval<0.10 else ''))
    print(f'  {label:<35} coef={coef:+.3f}  p={pval:.3f}  {sig}')

print()
# IRF comparison table — key channels at h=1,2
irf_v1 = v1.irf(periods=8)
lo_v1, hi_v1 = irf_v1.errband_mc(orth=True, repl=1000, signif=0.10)
si = {v: i for i, v in enumerate(ENDOG_OLD)}
ri = {v: i for i, v in enumerate(ENDOG_OLD)}
channels = [
    (si['hibor_3m'], ri['hk_property_price_qoq'], 'hibor → property'),
    (si['hk_exports_china_yoy'], ri['gdp_growth'], 'exports → gdp'),
    (si['gdp_growth'], ri['cpi_inflation'],         'gdp → cpi'),
]
print('Bootstrap IRF (90% MC, 1000 repl):')
print(f'{"Channel":<22} {"h":>3}  CI')
print('-'*55)
for imp, resp, label in channels:
    for h in [1, 2, 4]:
        lo = lo_v1[h, resp, imp]; hi = hi_v1[h, resp, imp]
        sig = 'sig' if not (lo < 0 < hi) else 'x0'
        print(f'{label:<22} {h:>3}  ({lo:+.3f}, {hi:+.3f}) {sig}')
    print()

# FEVD headline
fevd_v1 = v1.fevd(periods=8).decomp
print(f'FEVD at h=8: HIBOR→property={fevd_v1[ri["hk_property_price_qoq"]][7][si["hibor_3m"]]*100:.0f}%  |  exports→gdp={fevd_v1[ri["gdp_growth"]][7][si["hk_exports_china_yoy"]]*100:.0f}%')

# Save full IRF figure
fig = irf_v1.plot(orth=True, stderr_type='mc', repl=1000, signif=0.1)
fig.savefig('output/phase3_irf_bootstrap_full.png', dpi=80, bbox_inches='tight')
plt.close()
print('\nSaved: output/phase3_irf_bootstrap_full.png')

VARX(1) | obs=111 | AIC=4.3055 | BIC=5.6236

Ljung-Box (lag=8):
  hk_exports_china_yoy             p=0.0938  pass
  gdp_growth                       p=0.0000  FAIL
  cpi_inflation                    p=0.0052  FAIL
  unemployment                     p=0.1700  pass
  hibor_3m                         p=0.8839  pass
  hk_property_price_qoq            p=0.1056  pass

Exogenous coefficients (LERS + trade channel):
  us_ffr → hibor (LERS)               coef=+0.530  p=0.000  ***
  china_gdp → exports (trade)         coef=+0.806  p=0.016  **

Bootstrap IRF (90% MC, 1000 repl):
Channel                  h  CI
-------------------------------------------------------
hibor → property         1  (-1.605, -0.366) sig
hibor → property         2  (-0.924, -0.038) sig
hibor → property         4  (-0.302, +0.132) x0

exports → gdp            1  (+0.112, +0.814) sig
exports → gdp            2  (-0.176, +0.525) x0
exports → gdp            4  (-0.249, +0.251) x0

gdp → cpi                1  (+0.055, +0.357) 

## Phase 4 - BVAR(4) Minnesota Baseline Comparison

Shows BVAR(4) recovers the exports to gdp signal that VARX(4) destroyed via OLS overfitting.

Prior: pi1=0.1, pi2=0.5, pi3=1 (Litterman 1986 default)
Ordering: ENDOG_OLD (exports-first, for valid comparison with VARX(1))
ar_coefficients: AR_COEF_OLD [exports, gdp, cpi, unemp, hibor, property]

Key result: BVAR(4) exports to gdp at h=2: (+0.35, +0.57) sig vs VARX(1) crosses zero


In [6]:
# Phase 4: BVAR(4) Minnesota baseline — pi1=0.1, pi2=0.5, pi3=1 (Litterman default)

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y4 = df[ENDOG_OLD].values.astype(float)
X  = df[EXOG].values.astype(float)
idx4 = {v: i for i, v in enumerate(ENDOG_OLD)}

bvar4 = MinnesotaBayesianVar(
    endogenous=Y4, exogenous=X, lags=4,
    pi1=0.1, pi2=0.5, pi3=1,
    ar_coefficients=AR_COEF_OLD,
    iterations=2000, credibility_level=0.90, verbose=False
)
bvar4.estimate()
irf4, _ = bvar4.impulse_response_function(h=9, credibility_level=0.90)
fevd4   = bvar4.forecast_error_variance_decomposition(h=8, credibility_level=0.90)

channels = [
    (idx4['hibor_3m'], idx4['hk_property_price_qoq'], 'hibor → property'),
    (idx4['hk_exports_china_yoy'], idx4['gdp_growth'], 'exports → gdp'),
    (idx4['gdp_growth'], idx4['cpi_inflation'],         'gdp → cpi'),
]

# Reload VARX(1) IRF for side-by-side
v1_reload = VAR(df[ENDOG_OLD], exog=df[EXOG]).fit(maxlags=1)
irf_v1r   = v1_reload.irf(periods=8)
lo_v1r, hi_v1r = irf_v1r.errband_mc(orth=True, repl=500, signif=0.10)

print('Phase 4: IRF comparison — VARX(1) vs BVAR(4) Litterman (90% bands)')
print(f'{"Channel":<20} {"h":>3}  {"VARX(1)":>20}  {"BVAR(4)":>20}')
print('-'*70)
for imp, resp, label in channels:
    for h in [1, 2, 4]:
        v1_lo = lo_v1r[h, resp, imp]; v1_hi = hi_v1r[h, resp, imp]
        b4_lo = irf4[resp, imp, h, 1]; b4_hi = irf4[resp, imp, h, 2]
        v1_s  = 'sig' if not (v1_lo < 0 < v1_hi) else 'x0'
        b4_s  = 'sig' if not (b4_lo < 0 < b4_hi) else 'x0'
        print(f'{label:<20} {h:>3}  ({v1_lo:+.3f},{v1_hi:+.3f}){v1_s:>4}  ({b4_lo:+.3f},{b4_hi:+.3f}){b4_s:>4}')
    print()

print('FEVD at h=8 (BVAR(4) Litterman):')
print(f'  HIBOR share in property: {fevd4[idx4["hk_property_price_qoq"],idx4["hibor_3m"],7,0]*100:.0f}%')
print(f'  Exports share in GDP:    {fevd4[idx4["gdp_growth"],idx4["hk_exports_china_yoy"],7,0]*100:.0f}%')
print(f'  HIBOR own-share:         {fevd4[idx4["hibor_3m"],idx4["hibor_3m"],7,0]*100:.0f}%')

# Plot comparison
BLUE = '#1a6faf'; RED = '#c0392b'
t = np.arange(9)
fig, axes = plt.subplots(1, 3, figsize=(14, 4)); fig.patch.set_facecolor('white')
for ax, (imp, resp, title) in zip(axes, channels):
    ax.plot(t, irf_v1r.orth_irfs[:, resp, imp], color=BLUE, lw=2, label='VARX(1)')
    ax.fill_between(t, lo_v1r[:, resp, imp], hi_v1r[:, resp, imp], alpha=0.15, color=BLUE)
    ax.plot(t, irf4[resp, imp, :, 0], color=RED, lw=2, ls='--', label='BVAR(4)')
    ax.fill_between(t, irf4[resp, imp, :, 1], irf4[resp, imp, :, 2], alpha=0.15, color=RED)
    ax.axhline(0, color='k', lw=0.7, ls=':')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('Quarters', fontsize=8); ax.legend(fontsize=8); ax.tick_params(labelsize=8)
fig.suptitle('Phase 4: VARX(1) vs BVAR(4) Litterman (pi1=0.1) | exports→gdp recovered by BVAR', fontsize=9, y=1.01)
plt.tight_layout()
plt.savefig('output/phase4_bvar_irf_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/phase4_bvar_irf_comparison.png')

Phase 4: IRF comparison — VARX(1) vs BVAR(4) Litterman (90% bands)
Channel                h               VARX(1)               BVAR(4)
----------------------------------------------------------------------
hibor → property       1  (-1.595,-0.388) sig  (-0.631,-0.192) sig
hibor → property       2  (-0.908,-0.047) sig  (-0.341,+0.081)  x0
hibor → property       4  (-0.314,+0.126)  x0  (-0.142,+0.073)  x0

exports → gdp          1  (+0.139,+0.836) sig  (+0.285,+0.501) sig
exports → gdp          2  (-0.123,+0.591)  x0  (+0.106,+0.363) sig
exports → gdp          4  (-0.236,+0.270)  x0  (-0.141,+0.094)  x0

gdp → cpi              1  (+0.072,+0.347) sig  (+0.091,+0.179) sig
gdp → cpi              2  (+0.193,+0.505) sig  (+0.138,+0.258) sig
gdp → cpi              4  (+0.266,+0.687) sig  (+0.176,+0.320) sig

FEVD at h=8 (BVAR(4) Litterman):
  HIBOR share in property: 6%
  Exports share in GDP:    18%
  HIBOR own-share:         86%
Saved: output/phase4_bvar_irf_comparison.png


## Phase 5 - BVAR(4) Hyperparameter Optimization

5A: Lag-length ML grid, p=4, pi1=0.085, pi2=1.0 (ML-optimal)
5B: Posterior distribution verification (analytical draws, NOT MCMC)
5C: In-sample fit, 2/6 Ljung-Box fail (same as VARX(1), structural breaks)
5D: OOS RMSE, BVAR wins 18/18 cells

All cells use AR_COEF_OLD explicitly. Old runs used default delta=0.95 (bug fixed here).
pi2=1.0 universally selected, no cross-variable shrinkage.
pi4=99.75 from optimizer, consistent with uninformative prior on exogenous block.


In [7]:
# Phase 5A: Lag-length ML grid — ENDOG_OLD ordering (consistent with Phases 3-4)

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y5 = df[ENDOG_OLD].values.astype(float)
X  = df[EXOG].values.astype(float)

print('Lag-length grid — ML-optimized prior (iterations=500 for speed):')
print(f'{"p":>4}  {"opt_pi1":>8}  {"opt_pi2":>8}  {"ML":>12}  {"LB fail":>8}')
print('-' * 52)
opt_results = {}
for p in range(1, 6):
    bv = MinnesotaBayesianVar(
        endogenous=Y5, exogenous=X, lags=p,
        hyperparameter_optimization=True,
        iterations=500, credibility_level=0.90, verbose=False
    )
    bv.estimate()
    ml = bv.marginal_likelihood()
    bv.insample_fit()
    resid = bv.residual_estimates[:, :, 0]
    lb_fails = sum(
        acorr_ljungbox(resid[:, i], lags=[8], return_df=True)['lb_pvalue'].values[0] < 0.05
        for i in range(len(ENDOG_OLD))
    )
    opt_results[p] = {'ml': ml, 'pi1': bv.pi1, 'pi2': bv.pi2, 'lb': lb_fails}
    print(f'{p:>4}  {bv.pi1:>8.4f}  {bv.pi2:>8.4f}  {ml:>12.4f}  {lb_fails:>5}/6')

best_p = max(opt_results, key=lambda p: opt_results[p]['ml'])
print(f'\nML selects p={best_p} | pi1={opt_results[best_p]["pi1"]:.4f} | pi2={opt_results[best_p]["pi2"]:.4f}')
print(f'BVAR(4): ML={opt_results[4]["ml"]:.2f} | pi1={opt_results[4]["pi1"]:.4f} | LB={opt_results[4]["lb"]}/6')
print()
print('Note: p=3 optimizer may degenerate to pi1=0.01 (extremely tight prior — data ignored).')
print('      BVAR(4) is the best joint ML+diagnostic model (pi1~0.085, 2/6 LB fail).')
print()
print(f'FINAL SPEC confirmed: p=4, pi1={PI1}, pi2={PI2}, pi3={PI3}')
print(f'AR_COEF_OLD (ML-optimal delta): {AR_COEF_OLD}  [exp, gdp, cpi, unemp, hibor, prop]')

Lag-length grid — ML-optimized prior (iterations=500 for speed):
   p   opt_pi1   opt_pi2            ML   LB fail
----------------------------------------------------
   1    0.0466    1.0000     -534.2393      3/6
   2    0.0407    1.0000     -534.4678      3/6
   3    0.0100    1.0000     -523.3593      4/6
   4    0.0846    1.0000     -528.9796      2/6
   5    0.1030    1.0000     -528.0299      2/6

ML selects p=3 | pi1=0.0100 | pi2=1.0000
BVAR(4): ML=-528.98 | pi1=0.0846 | LB=2/6

Note: p=3 optimizer may degenerate to pi1=0.01 (extremely tight prior — data ignored).
      BVAR(4) is the best joint ML+diagnostic model (pi1~0.085, 2/6 LB fail).

FINAL SPEC confirmed: p=4, pi1=0.085, pi2=1.0, pi3=1.0
AR_COEF (ML-optimal delta): [0.627, 0.545, 0.735, 0.991, 0.442, 0.418]  [exp, gdp, cpi, unemp, hibor, prop]


In [8]:
# Phase 5B: Posterior distribution verification
# NOT MCMC convergence — Minnesota BVAR has closed-form Normal posterior.
# Draws are i.i.d. from exact posterior. Trace/ESS are category errors here.
# We verify: signs correct, posteriors unimodal, spreads plausible.

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y5 = df[ENDOG_OLD].values.astype(float)
X  = df[EXOG].values.astype(float)

bv5b = MinnesotaBayesianVar(
    endogenous=Y5, exogenous=X, lags=LAGS,
    pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_OLD,
    iterations=5000, credibility_level=0.90, verbose=False
)
bv5b.estimate()
draws = bv5b.mcmc_beta  # (k, n, iterations) — i.i.d. analytical draws

# Monitor key parameters: (reg_i, eq_i, label, expected_sign)
# Regressor order: const, then exogenous, then L1.endog, L2.endog, ...
# Indices approximate — depend on exact alexandria internal ordering
monitor = [
    (1, 4, 'us_ffr → hibor',        '+'),
    (2, 0, 'china_gdp → exports',   '+'),
    (3, 1, 'L1.exports → gdp',      '+'),
    (4, 2, 'L1.gdp → cpi',          '+'),
    (7, 5, 'L1.hibor → property',   '-'),
    (0, 1, 'const → gdp',           '?'),
]

print('Posterior Verification (5000 i.i.d. analytical draws):')
print(f'{"Parameter":<28} {"Mean":>8} {"Std":>8} {"Sign OK":>8}')
print('-' * 58)
for reg_i, eq_i, label, exp_sign in monitor:
    chain = draws[reg_i, eq_i, :]
    mean = chain.mean(); std = chain.std()
    if exp_sign == '+': sign_ok = 'YES' if mean > 0 else 'FAIL'
    elif exp_sign == '-': sign_ok = 'YES' if mean < 0 else 'FAIL'
    else: sign_ok = 'N/A'
    print(f'{label:<28} {mean:>8.4f} {std:>8.4f} {sign_ok:>8}')

# Density plots
fig, axes = plt.subplots(2, 3, figsize=(13, 7)); fig.patch.set_facecolor('white')
for ax, (reg_i, eq_i, label, exp_sign) in zip(axes.flat, monitor):
    chain = draws[reg_i, eq_i, :]
    ax.hist(chain, bins=60, color='#1a6faf', alpha=0.7, density=True)
    ax.axvline(chain.mean(), color='red', lw=1.5, ls='--', label=f'mean={chain.mean():.3f}')
    ax.axvline(0, color='black', lw=0.8, ls=':')
    ax.set_title(f'{label} (exp: {exp_sign})', fontsize=8, fontweight='bold')
    ax.set_ylabel('Density', fontsize=7); ax.tick_params(labelsize=7); ax.legend(fontsize=7)
fig.suptitle('Phase 5B: Posterior Verification — BVAR(4) pi1=0.085 pi2=1.0 AR_COEF_OLD=ML-optimal\nAnalytical draws from closed-form Normal posterior. Not MCMC.', fontsize=9, y=1.01)
plt.tight_layout()
plt.savefig('output/phase5_posterior_verification.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/phase5_posterior_verification.png')

Posterior Verification (5000 i.i.d. analytical draws):
Parameter                        Mean      Std  Sign OK
----------------------------------------------------------
us_ffr → hibor                 0.5374   0.0588      YES
china_gdp → exports            0.8400   0.2448      YES
L1.exports → gdp              -0.0002   0.0135     FAIL
L1.gdp → cpi                   0.0310   0.0234      YES
L1.hibor → property           -0.5443   0.4003      YES
const → gdp                   -2.1976   0.9504      N/A
Saved: output/phase5_posterior_verification.png


In [9]:
# Phase 5C: In-sample fit diagnostics

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y5 = df[ENDOG_OLD].values.astype(float)
X  = df[EXOG].values.astype(float)

bv5c = MinnesotaBayesianVar(
    endogenous=Y5, exogenous=X, lags=LAGS,
    pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_OLD,
    iterations=2000, credibility_level=0.90, verbose=False
)
bv5c.estimate()
bv5c.insample_fit()
resid5 = bv5c.residual_estimates[:, :, 0]
T_eff  = resid5.shape[0]
Y_tr   = Y5[-T_eff:]
fitted = Y_tr - resid5

print(f'In-sample fit — BVAR(4) pi1={PI1} pi2={PI2} AR_COEF_OLD=ML-optimal:')
print(f'{"Variable":<32} {"R2":>6} {"RMSE":>8} {"LB p":>8} {"Status":>8}')
print('-' * 65)
for i, col in enumerate(ENDOG_OLD):
    ss_res = np.sum(resid5[:, i]**2)
    ss_tot = np.sum((Y_tr[:, i] - Y_tr[:, i].mean())**2)
    r2   = 1 - ss_res / ss_tot
    rmse = np.sqrt(ss_res / T_eff)
    lb_p = acorr_ljungbox(resid5[:, i], lags=[8], return_df=True)['lb_pvalue'].values[0]
    print(f'{col:<32} {r2:>6.3f} {rmse:>8.3f} {lb_p:>8.4f} {"FAIL" if lb_p<0.05 else "pass":>8}')

# Plot fitted vs actual for key variables
focus = ['gdp_growth', 'cpi_inflation', 'hibor_3m']
fig, axes = plt.subplots(3, 3, figsize=(14, 9)); fig.patch.set_facecolor('white')
for row, col in enumerate(focus):
    i = ENDOG_OLD.index(col)
    ax0, ax1, ax2 = axes[row]
    ax0.plot(Y_tr[:, i], 'k-', lw=1.2, label='Actual')
    ax0.plot(fitted[:, i], '#c0392b', lw=1.2, ls='--', label='Fitted')
    ax0.set_title(f'{col} — fitted vs actual', fontsize=8, fontweight='bold'); ax0.legend(fontsize=7)
    ax1.plot(resid5[:, i], '#1a6faf', lw=0.8); ax1.axhline(0, color='k', lw=0.5, ls=':')
    ax1.set_title('Residuals', fontsize=8)
    plot_acf(resid5[:, i], lags=16, ax=ax2, alpha=0.05); ax2.set_title('ACF', fontsize=8)
    for ax in [ax0, ax1, ax2]: ax.tick_params(labelsize=7)
plt.suptitle('Phase 5C: In-sample fit — BVAR(4) pi1=0.085 pi2=1.0 (ML-optimal AR_COEF_OLD)\n2/6 LB fail = structural breaks, not lag misspecification', fontsize=9, y=1.01)
plt.tight_layout()
plt.savefig('output/phase5_insample_fit.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved: output/phase5_insample_fit.png')

In-sample fit — BVAR(4) pi1=0.085 pi2=1.0 AR_COEF=ML-optimal:
Variable                             R2     RMSE     LB p   Status
-----------------------------------------------------------------
hk_exports_china_yoy              0.644    7.352   0.2444     pass
gdp_growth                        0.788    1.801   0.0000     FAIL
cpi_inflation                     0.915    0.746   0.0026     FAIL
unemployment                      0.948    0.332   0.6243     pass
hibor_3m                          0.954    0.406   0.2743     pass
hk_property_price_qoq             0.411    3.419   0.3881     pass
Saved: output/phase5_insample_fit.png


In [10]:
# Phase 5D: Out-of-sample RMSE comparison — expanding window
# Monkey-patch fixes numpy [] vs array comparison bug in alexandria v3.0.0

def _fixed_forecast_regressors(Z_p, Y, h, p, T, exogenous, constant, trend, quadratic_trend):
    temp = vu.generate_intercept_and_trends(constant, trend, quadratic_trend, h, T)
    exog_empty = (exogenous is None or
                  (isinstance(exogenous, list) and len(exogenous) == 0) or
                  (isinstance(exogenous, np.ndarray) and exogenous.size == 0))
    if exog_empty:
        Z_p = []
    elif isinstance(Z_p, list) and len(Z_p) == 0:
        Z_p = np.tile(exogenous[-1], [h, 1])
    if len(Z_p) != 0:
        Z_p = np.hstack([temp, Z_p])
    elif any([constant, trend, quadratic_trend]):
        Z_p = temp
    else:
        Z_p = []
    Y = Y[-p:, :]
    return Z_p, Y
vu.make_forecast_regressors = _fixed_forecast_regressors

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
Y_all = df[ENDOG_OLD].values.astype(float)
X_all = df[EXOG].values.astype(float)
T = len(Y_all); T_start = 85; horizons = [1, 2, 4]

v1_preds = {h: [] for h in horizons}
bv_preds = {h: [] for h in horizons}
actuals  = {h: [] for h in horizons}

t0 = time.time()
for t in range(T_start, T):
    df_tr = df.iloc[:t]; Y_tr = Y_all[:t]; X_tr = X_all[:t]
    m1 = VAR(df_tr[ENDOG_OLD], exog=df_tr[EXOG]).fit(maxlags=1)
    bv = MinnesotaBayesianVar(
        endogenous=Y_tr, exogenous=X_tr, lags=LAGS,
        pi1=PI1, pi2=PI2, pi3=PI3,
        ar_coefficients=AR_COEF_OLD,
        iterations=300, credibility_level=0.90, verbose=False
    )
    bv.estimate()
    for h in horizons:
        if t + h > T: continue
        X_fut = X_all[t:t+h]
        fc1   = m1.forecast(df_tr[ENDOG_OLD].values[-1:], steps=h, exog_future=X_fut)
        fc_bv = bv.forecast(h=h, credibility_level=0.90, Z_p=X_fut)
        v1_preds[h].append(fc1[h-1])
        bv_preds[h].append(fc_bv[h-1, :, 0])
        actuals[h].append(Y_all[t+h-1])
print(f'OOS done: {time.time()-t0:.1f}s | {T-T_start} windows')

print('\nOOS RMSE ratio (BVAR/VARX1 — <1 = BVAR wins):')
print(f'{"Variable":<28} {"h=1":>8}  {"h=2":>8}  {"h=4":>8}')
print('-' * 58)
wins = 0
for i, col in enumerate(ENDOG_OLD):
    ratios = []
    for h in horizons:
        act = np.array(actuals[h])[:,i]
        rv1 = np.sqrt(np.nanmean((np.array(v1_preds[h])[:,i]-act)**2))
        rbv = np.sqrt(np.nanmean((np.array(bv_preds[h])[:,i]-act)**2))
        r   = rbv/rv1
        ratios.append(r)
        if r < 1: wins += 1
    stars = ['**' if r < 1 else '  ' for r in ratios]
    print(f'{col:<28} {ratios[0]:>6.3f}{stars[0]}  {ratios[1]:>6.3f}{stars[1]}  {ratios[2]:>6.3f}{stars[2]}')
print(f'\nBVAR wins {wins}/18 cells (** = BVAR wins)')

OOS done: 0.6s | 27 windows

OOS RMSE ratio (BVAR/VARX1 — <1 = BVAR wins):
Variable                          h=1       h=2       h=4
----------------------------------------------------------
hk_exports_china_yoy          0.872**   0.912**   0.930**
gdp_growth                    0.900**   0.905**   0.885**
cpi_inflation                 0.969**   0.921**   0.800**
unemployment                  0.962**   0.916**   0.814**
hibor_3m                      0.842**   0.808**   0.866**
hk_property_price_qoq         0.872**   0.920**   0.967**

BVAR wins 18/18 cells (** = BVAR wins)


## Phase 7 - Stationarity Audit and Delta-u Robustness

ADF+KPSS results:

| Series | ADF_p | KPSS_p | Verdict |
|---|---|---|---|
| hk_exports_china_yoy | 0.000 | 0.100 | I(0) |
| gdp_growth | 0.018 | 0.100 | I(0) |
| cpi_inflation | 0.171 | 0.016 | I(1) |
| unemployment | 0.091 | 0.010 | I(1) |
| hibor_3m | 0.000 | 0.045 | ambiguous (LERS mean-reversion) |
| hk_property_price_qoq | 0.000 | 0.099 | I(0) |

Bayesian defense of levels: Sims, Stock and Watson (1990). The prior regularizes the coefficient space. BVAR-in-levels is valid.

Delta-u robustness: Substituting delta-u for unemployment leaves headline IRFs unchanged. Keep levels.


In [11]:
# Phase 7.2: ADF + KPSS stationarity battery

df = pd.read_csv(DATA, index_col=0, parse_dates=True)
prop_idx = df['hk_property_price_idx']

test_series = {
    'hk_exports_china_yoy':  df['hk_exports_china_yoy'],
    'gdp_growth':            df['gdp_growth'],
    'cpi_inflation':         df['cpi_inflation'],
    'unemployment':          df['unemployment'],
    'hibor_3m':              df['hibor_3m'],
    'hk_property_price_qoq': df['hk_property_price_qoq'],
    'hk_property_price_idx': prop_idx,
}

print('ADF + KPSS stationarity audit:')
print(f'{"Series":<28} {"ADF_p":>8}  {"KPSS_p":>8}  Verdict')
print('-' * 65)
i1_candidates = []
for name, s in test_series.items():
    s_clean = s.dropna()
    adf_p = adfuller(s_clean, autolag='BIC')[1]
    try:
        kpss_stat, kpss_p, _, _ = kpss(s_clean, regression='c', nlags='auto')
        kpss_p_display = kpss_p
    except Exception:
        kpss_p_display = np.nan
    # I(1) if both tests agree: ADF fails to reject AND KPSS rejects
    if adf_p > 0.05 and (np.isnan(kpss_p_display) or kpss_p_display < 0.05):
        verdict = 'I(1)'
        i1_candidates.append(name)
    elif adf_p < 0.05 and (np.isnan(kpss_p_display) or kpss_p_display > 0.05):
        verdict = 'I(0)'
    else:
        verdict = 'ambiguous'
    print(f'{name:<28} {adf_p:>8.4f}  {kpss_p_display if not np.isnan(kpss_p_display) else "nan":>8}  {verdict}')

print(f'\nI(1) candidates: {i1_candidates}')
print('Endogenous I(1) block for Johansen: [unemployment, cpi_inflation, hk_property_price_idx]')

# Phase 7.2c: Δu robustness
print('\n--- Phase 7.2c: Δu robustness ---')

df_du = df.copy()
df_du['unemployment'] = df['unemployment'].diff()
df_du = df_du.dropna()

bvar_base = MinnesotaBayesianVar(
    endogenous=df[ENDOG_OLD].values, exogenous=df[EXOG].values,
    lags=LAGS, pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_OLD,
    iterations=3000, credibility_level=0.90, verbose=False
)
bvar_base.estimate()
irf_base, _ = bvar_base.impulse_response_function(h=9, credibility_level=0.90)

bvar_du = MinnesotaBayesianVar(
    endogenous=df_du[ENDOG_OLD].values, exogenous=df_du[EXOG].values,
    lags=LAGS, pi1=PI1, pi2=PI2, pi3=PI3,
    ar_coefficients=AR_COEF_OLD,
    iterations=3000, credibility_level=0.90, verbose=False
)
bvar_du.estimate()
irf_du, _ = bvar_du.impulse_response_function(h=9, credibility_level=0.90)

idx4 = {v: i for i, v in enumerate(ENDOG_OLD)}
checks = [('HIBOR → Property', idx4['hibor_3m'], idx4['hk_property_price_qoq']),
          ('Exports → GDP',    idx4['hk_exports_china_yoy'], idx4['gdp_growth'])]

print('IRF at h=2 — baseline vs Δu robustness:')
print(f'{"Channel":<22} {"Base CI":>22}  {"Δu CI":>22}  Verdict')
print('-' * 75)
for label, imp, resp in checks:
    b_lo = irf_base[resp, imp, 2, 1]; b_hi = irf_base[resp, imp, 2, 2]
    d_lo = irf_du[resp, imp, 2, 1];   d_hi = irf_du[resp, imp, 2, 2]
    b_s  = 'sig' if not (b_lo < 0 < b_hi) else 'x0'
    d_s  = 'sig' if not (d_lo < 0 < d_hi) else 'x0'
    verdict = 'PASS' if b_s == d_s else 'CHANGED'
    print(f'{label:<22} ({b_lo:+.3f},{b_hi:+.3f}) {b_s}  ({d_lo:+.3f},{d_hi:+.3f}) {d_s}  {verdict}')

print('\nConclusion: Keep unemployment in levels. One limitation sentence required.')

ADF + KPSS stationarity audit:
Series                          ADF_p    KPSS_p  Verdict
-----------------------------------------------------------------
hk_exports_china_yoy           0.0000       0.1  I(0)
gdp_growth                     0.0179       0.1  I(0)
cpi_inflation                  0.1713  0.01565285079847136  I(1)
unemployment                   0.0911      0.01  I(1)
hibor_3m                       0.0003  0.04545284540694442  ambiguous
hk_property_price_qoq          0.0000  0.09924875174854104  I(0)
hk_property_price_idx          0.8210      0.01  I(1)

I(1) candidates: ['cpi_inflation', 'unemployment', 'hk_property_price_idx']
Endogenous I(1) block for Johansen: [unemployment, cpi_inflation, hk_property_price_idx]

--- Phase 7.2c: Δu robustness ---
IRF at h=2 — baseline vs Δu robustness:
Channel                               Base CI                   Δu CI  Verdict
---------------------------------------------------------------------------
HIBOR → Property       (-0.495,+0.

## Phase 8.4 - Johansen Cointegration

Test block: unemployment, cpi_inflation, hk_property_price_idx (I(1) endogenous only)
Exogenous excluded: LERS peg mechanics are not cointegration.

| Test | Statistics | 95% CV | Rank |
|---|---|---|---|
| Trace | [24.57, 9.56, 0.75] | [29.8, 15.49, 3.84] | 0 |
| Max-eigenvalue | [15.0, 8.82, 0.75] | [21.13, 14.26, 3.84] | 0 |

Rank=0 at 95%. VECM not warranted. BVAR-in-levels is appropriate.


In [12]:
# Phase 8.4: Johansen cointegration — endogenous I(1) block only

df    = pd.read_csv(DATA, index_col=0, parse_dates=True)
prop_idx = df['hk_property_price_idx']

# I(1) endogenous block — exogenous excluded deliberately
i1_block = pd.DataFrame({
    'unemployment':         df['unemployment'],
    'cpi_inflation':        df['cpi_inflation'],
    'hk_property_price_idx': prop_idx,
}).dropna()

print(f'I(1) endogenous block: n={len(i1_block)} obs, variables: {list(i1_block.columns)}')
print('Note: us_ffr and china_gdp EXCLUDED — LERS peg is not cointegration')
print()

res = coint_johansen(i1_block, det_order=0, k_ar_diff=1)
print('Johansen results:')
print(f'  Trace statistics:      {[round(x,2) for x in res.lr1]}')
print(f'  Trace CV 95%:          {[round(x,2) for x in res.cvt[:, 1]]}')
print(f'  Max-eigenvalue stats:  {[round(x,2) for x in res.lr2]}')
print(f'  Max-eig CV 95%:        {[round(x,2) for x in res.cvm[:, 1]]}')
print()

# Determine rank
rank = 0
for i, (stat, cv) in enumerate(zip(res.lr1, res.cvt[:, 1])):
    if stat > cv:
        rank = i + 1
print(f'Cointegrating rank at 95%: {rank}')
if rank == 0:
    print('VECM not warranted. BVAR-in-levels is the appropriate framework.')
    print('This is a research finding: I(1) variables in this system do not form stable long-run relationships.')
else:
    print(f'Rank={rank} — consider BVECM. Check if driven by exogenous contamination.')

I(1) endogenous block: n=112 obs, variables: ['unemployment', 'cpi_inflation', 'hk_property_price_idx']
Note: us_ffr and china_gdp EXCLUDED — LERS peg is not cointegration

Johansen results:
  Trace statistics:      [np.float64(26.68), np.float64(9.97), np.float64(0.8)]
  Trace CV 95%:          [np.float64(29.8), np.float64(15.49), np.float64(3.84)]
  Max-eigenvalue stats:  [np.float64(16.71), np.float64(9.17), np.float64(0.8)]
  Max-eig CV 95%:        [np.float64(21.13), np.float64(14.26), np.float64(3.84)]

Cointegrating rank at 95%: 0
VECM not warranted. BVAR-in-levels is the appropriate framework.
This is a research finding: I(1) variables in this system do not form stable long-run relationships.
